# 손글씨 숫자 인식 — 베이스라인 학습 노트북

이 노트북은 **그대로 위에서 아래로 실행하면 동작하는** 베이스라인입니다.

여러분의 과제는 학습 코드를 새로 짜는 것이 아니라:

1. 베이스라인을 실행해 **정확도를 확인**하고 (`model.pt` 저장까지)
2. 맨 아래 **TODO**를 따라 정확도를 올리고, **실험 기록 표**를 채우고
3. 저장된 모델을 `server.py` + `app.py`(그림판)로 **서빙**하는 것입니다.

> 완성 기준(2일차): 테스트 정확도 97% 이상 + `model.pt` 저장 + 실험 기록 2건 이상

## 1. 준비 — 라이브러리와 디바이스 확인

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)   # cpu 여도 충분히 돌아갑니다

## 2. 데이터 불러오기

MNIST는 `torchvision`이 자동으로 내려받습니다 (약 11MB, 처음 한 번만).

- `ToTensor()`: 픽셀을 0~1 텐서로
- `Normalize(평균, 표준편차)`: 학습 안정화 — **서빙(server.py)에서도 같은 값을 쓰므로 바꾸지 마세요**

In [ ]:
from model import MNIST_MEAN, MNIST_STD

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((MNIST_MEAN,), (MNIST_STD,)),
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

print("train:", len(train_ds), "/ test:", len(test_ds))

In [ ]:
# 데이터가 어떻게 생겼는지 눈으로 확인
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img, label in zip(axes, images, labels):
    ax.imshow(img.squeeze(), cmap="gray")
    ax.set_title(int(label))
    ax.axis("off")
plt.show()

## 3. 모델 만들기

모델 구조는 `model.py`의 `SmallCNN`에 있습니다. **노트북과 서버가 같은 파일을 import** 하기 때문에,
구조를 바꾸고 싶으면 `model.py`를 수정한 뒤 이 노트북에서 다시 학습·저장하면 됩니다.

In [ ]:
from model import SmallCNN

model = SmallCNN().to(device)
print(model)

## 4. 학습

베이스라인은 1 epoch만 돕니다 (CPU 기준 2~3분). 그래도 정확도 97% 근처가 나옵니다.

In [ ]:
EPOCHS = 1     # TODO: 3~5로 늘려보세요
LR = 1e-3      # TODO: 1e-2, 1e-4 로 바꾸면 어떻게 될까?

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        if (i + 1) % 200 == 0:
            print(f"epoch {epoch+1} | step {i+1:4d}/{len(train_loader)} | loss {running_loss/200:.4f}")
            running_loss = 0.0

print("학습 완료")

## 5. 평가 — 테스트 정확도

이 숫자가 여러분의 **KPI**입니다. 실험할 때마다 아래 실험 기록 표에 적으세요.

In [ ]:
model.eval()
correct = 0
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()

accuracy = correct / len(test_ds)
print(f"테스트 정확도: {accuracy:.4f} ({correct}/{len(test_ds)})")

## 6. 모델 저장 — 서빙 준비

`server.py`가 시작할 때 이 파일(`model.pt`)을 로드합니다.

In [ ]:
torch.save(model.state_dict(), "model.pt")
print("model.pt 저장 완료 — 이제 터미널에서 서버를 띄울 수 있습니다")
print("  uvicorn server:app --reload --port 8000")

In [ ]:
# 저장이 제대로 됐는지 다시 불러와 확인 (서버가 하는 일과 동일)
loaded = SmallCNN()
loaded.load_state_dict(torch.load("model.pt", map_location="cpu"))
loaded.eval()

x, y = test_ds[0]
with torch.no_grad():
    probs = torch.softmax(loaded(x.unsqueeze(0)), dim=1)[0]
print("정답:", y, "/ 예측:", int(probs.argmax()), f"/ 확신도: {float(probs.max()):.3f}")

---

## TODO — 정확도 올리기 (하나 바꿀 때마다 기록!)

한 번에 하나만 바꾸고 4~5단계를 다시 실행하세요. 그래야 뭐가 효과 있었는지 알 수 있습니다.

- [ ] TODO 1. `EPOCHS`를 3~5로 늘리기
- [ ] TODO 2. `LR`을 바꿔보기 (1e-2 / 1e-4) — 커지면? 작아지면?
- [ ] TODO 3. `model.py`에 `nn.Dropout(0.3)` 추가하기
- [ ] TODO 4. (심화) 데이터 증강: `transforms.RandomRotation(10)` 을 train transform에 추가
- [ ] TODO 5. (심화) 잘못 맞춘 이미지 10개를 모아 시각화하고, 왜 틀렸을지 분석

## 실험 기록 (발표 자료에 그대로 들어갑니다)

| # | 바꾼 것 | 테스트 정확도 | 메모 |
|---|---|---|---|
| 0 | baseline (epoch 1, lr 1e-3) | 0.____ | |
| 1 |  |  |  |
| 2 |  |  |  |
| 3 |  |  |  |

다 됐으면 → `README.md`의 순서대로 서버와 그림판 UI를 띄워 데모를 완성하세요.